In [ ]:
import pandas as pd
import sqlite3

print("1. Lendo os arquivos CSV limpos...")
df_acidentes = pd.read_csv("dados/acidentes_transito_limpo.csv")
df_obitos = pd.read_csv("dados/obitos_gerais_bh_2021.csv")

# Uma pequena preparação: Para o SQL conseguir cruzar os dados, precisamos que 
# a tabela de acidentes tenha uma coluna 'Mes' escrita exatamente igual à do DataSUS
df_acidentes['DATA_REAL'] = pd.to_datetime(df_acidentes['DATA HORA_BOLETIM'], format='%d/%m/%Y %H:%M')
meses_pt = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Marco', 4: 'Abril', 5: 'Maio', 6: 'Junho', 
            7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}
df_acidentes['Mes'] = df_acidentes['DATA_REAL'].dt.month.map(meses_pt)


print("2. Criando o Banco de Dados SQL...")
# Isso vai criar um arquivo chamado 'projeto_transito.db' na sua pasta
conexao = sqlite3.connect('projeto_transito.db')

print("3. Injetando as tabelas no Banco de Dados...")
# O comando .to_sql transforma o DataFrame do Pandas em uma Tabela real de SQL
df_acidentes.to_sql('tabela_acidentes', conexao, if_exists='replace', index=False)
df_obitos.to_sql('tabela_obitos', conexao, if_exists='replace', index=False)

print("✅ Sucesso! O Banco de Dados está pronto para receber consultas.")

1. Lendo os arquivos CSV limpos...
2. Criando o Banco de Dados SQL...
3. Injetando as tabelas no Banco de Dados...
✅ Sucesso! O Banco de Dados está pronto para receber consultas.


In [ ]:
# Escrevemos a consulta em formato de texto (String)
query_teste = """
    SELECT *
    FROM tabela_acidentes
    LIMIT 5;
"""

# O Pandas vai até o banco, executa a query e traz o resultado formatado
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

       NUMERO_BOLETIM DATA HORA_BOLETIM     DATA_INCLUSAO TIPO_ACIDENTE  \
0  2021-008886628-002  20/02/2021 10:01  20/02/2021 11:10        H01002   
1  2021-008888878-001  20/02/2021 10:25  20/02/2021 11:30        H09002   
2  2021-008891464-001  20/02/2021 11:22  20/02/2021 11:55        H04000   
3  2021-008891884-001  19/02/2021 23:00  20/02/2021 11:59        H08002   
4  2021-008892064-001  20/02/2021 11:22  20/02/2021 12:02        H01002   

                                  DESC_TIPO_ACIDENTE  COD_TEMPO  \
0  ABALROAMENTO COM VITIMA                       ...          1   
1  COLISAO DE VEICULOS COM VITIMA                ...          1   
2  QUEDA DE PESSOA DE VEICULO                    ...          1   
3  CHOQUE MECANICO COM VITIMA                    ...          0   
4  ABALROAMENTO COM VITIMA                       ...          1   

        DESC_TEMPO  COD_PAVIMENTO        PAVIMENTO  COD_REGIONAL  ...  \
0  BOM                          1  ASFALTO                    21  ...   


In [5]:
# Escrevemos a consulta em formato de texto (String)
query_teste = """
    SELECT *
    FROM tabela_obitos;
"""

# O Pandas vai até o banco, executa a query e traz o resultado formatado
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

    Ano do Óbito        Mes  Total_Obitos_Gerais
0           2021    Janeiro                 1874
1           2021  Fevereiro                 1544
2           2021      Marco                 2551
3           2021      Abril                 2553
4           2021       Maio                 2072
5           2021      Junho                 1845
6           2021      Julho                 1841
7           2021     Agosto                 1587
8           2021   Setembro                 1445
9           2021    Outubro                 1445
10          2021   Novembro                 1364
11          2021   Dezembro                 1486


Funções SQL

In [ ]:
query_teste = """
    SELECT COUNT(*) as 'Total de acidetes' FROM tabela_acidentes;
"""
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

   Total de acidetes
0              11122


In [11]:
query_teste = """
    SELECT SUM(Total_Obitos_Gerais) as 'Total_Obitos_Gerais' FROM tabela_obitos;
"""
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

   Total_Obitos_Gerais
0                21607


In [ ]:
query_teste = """
    SELECT DESC_TIPO_ACIDENTE, COUNT(*) as Total_Acidentes
FROM tabela_acidentes
GROUP BY DESC_TIPO_ACIDENTE
ORDER BY Total_Acidentes DESC;
"""
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

                                   DESC_TIPO_ACIDENTE  Total_Acidentes
0   ABALROAMENTO COM VITIMA                       ...             5079
1   CHOQUE MECANICO COM VITIMA                    ...             2071
2   COLISAO DE VEICULOS COM VITIMA                ...             1609
3   ATROPELAMENTO DE PESSOA SEM VITIMA FATAL      ...             1046
4   CAPOTAMENTO/TOMBAMENTO COM VITIMA             ...              833
5   QUEDA DE PESSOA DE VEICULO                    ...              279
6   OUTROS COM VITIMA                             ...              110
7   ATROPELAMENTO DE ANIMAL COM VITIMA            ...               42
8   ATROPELAMENTO DE PESSOA COM VITIMA FATAL      ...               36
9   QUEDA DE VEICULO COM VITIMA                   ...               13
10  QUEDA E/OU VAZAMENTO DE CARGA DE VEICULO C/ VI...                4


In [14]:

query_teste = """
    SELECT ROUND(AVG(Total_Obitos_Gerais), 2) FROM tabela_obitos;
"""
resultado = pd.read_sql_query(query_teste, conexao)
print(resultado)

   ROUND(AVG(Total_Obitos_Gerais), 2)
0                             1800.58


Abaixo apresento as consultas desenvolvidas para aprofundar a análise descritiva dos dados. O objetivo principal destas queries foi cruzar as informações dos múltiplos datasets para isolar métricas de letalidade, dimensionando estatisticamente o perigo do trânsito na região frente às demais causas de mortalidade.

In [ ]:
import pandas as pd
import sqlite3

# Conecta ao nosso banco de dados
conexao = sqlite3.connect('projeto_transito.db')

# ==============================================================================
# Qual é o tipo de acidente mais LETAL? 
# (Usa Agregação Avançada e Matemática)
# ==============================================================================
query_letalidade = """
    SELECT 
        DESC_TIPO_ACIDENTE AS Tipo_Acidente, COUNT(*) AS Total_Ocorrencias,
        SUM(CASE WHEN INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) AS Acidentes_Fatais,
        ROUND(
            (SUM(CASE WHEN INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 
        2) AS Taxa_Letalidade_Percentual
    FROM tabela_acidentes
    GROUP BY DESC_TIPO_ACIDENTE
    HAVING Total_Ocorrencias > 50
    ORDER BY Taxa_Letalidade_Percentual DESC;
"""

print("--- RANKING DE LETALIDADE POR TIPO DE ACIDENTE ---")
resultado_letalidade = pd.read_sql_query(query_letalidade, conexao)
print(resultado_letalidade)
print("\n" + "="*60 + "\n")

--- RANKING DE LETALIDADE POR TIPO DE ACIDENTE ---
                                       Tipo_Acidente  Total_Ocorrencias  \
0  OUTROS COM VITIMA                             ...                110   
1  CHOQUE MECANICO COM VITIMA                    ...               2071   
2  COLISAO DE VEICULOS COM VITIMA                ...               1609   
3  CAPOTAMENTO/TOMBAMENTO COM VITIMA             ...                833   
4  ABALROAMENTO COM VITIMA                       ...               5079   
5  QUEDA DE PESSOA DE VEICULO                    ...                279   
6  ATROPELAMENTO DE PESSOA SEM VITIMA FATAL      ...               1046   

   Acidentes_Fatais  Taxa_Letalidade_Percentual  
0                 8                        7.27  
1                28                        1.35  
2                 8                        0.50  
3                 4                        0.48  
4                21                        0.41  
5                 1                        0.36 

In [ ]:
# ==============================================================================
# Aplicando JOIN 
# Qual a proporção de mortes no trânsito em relação aos óbitos gerais por mês?
# ==============================================================================
query_cruzamento = """
    SELECT 
        a.Mes,
        COUNT(a.NUMERO_BOLETIM) AS Total_Acidentes,
        SUM(CASE WHEN a.INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) AS Obitos_Transito,
        o.Total_Obitos_Gerais,
        CONCAT(ROUND(
            (SUM(CASE WHEN a.INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) * 100.0) / o.Total_Obitos_Gerais, 
        2), '%') AS Perc_Transito_vs_Mortalidade
    FROM tabela_acidentes a
    INNER JOIN tabela_obitos o ON a.Mes = o.Mes
    GROUP BY a.Mes, o.Total_Obitos_Gerais
    ORDER BY Total_Acidentes DESC;
"""

print("--- CRUZAMENTO: TRÂNSITO VS MORTALIDADE GERAL (POR MÊS) ---")
resultado_cruzamento = pd.read_sql_query(query_cruzamento, conexao)
print(resultado_cruzamento)

--- CRUZAMENTO: TRÂNSITO VS MORTALIDADE GERAL (POR MÊS) ---
          Mes  Total_Acidentes  Obitos_Transito  Total_Obitos_Gerais  \
0      Agosto             1051               13                 1587   
1       Julho             1040               11                 1841   
2    Dezembro             1000                4                 1486   
3    Novembro              984                7                 1364   
4    Setembro              969                8                 1445   
5        Maio              942                7                 2072   
6     Outubro              935               10                 1445   
7       Junho              929               10                 1845   
8     Janeiro              844               10                 1874   
9   Fevereiro              832                7                 1544   
10      Abril              805               11                 2553   
11      Marco              791                9                 2551   

   

In [ ]:
import pandas as pd
import sqlite3

# Conecta ao nosso banco de dados
conexao = sqlite3.connect('projeto_transito.db')

# ==============================================================================
# Análise de Acidentes e Letalidade por Dia da Semana
# (Usa funções de Data/Hora do SQL e tradução condicional)
# ==============================================================================
query_dias_semana = """
    SELECT 
        -- A função strftime('%w', data) retorna 0 para Domingo, 1 para Segunda...
        CASE CAST(strftime('%w', DATA_REAL) AS INTEGER)
            WHEN 0 THEN '7-Domingo'
            WHEN 1 THEN '1-Segunda-feira'
            WHEN 2 THEN '2-Terça-feira'
            WHEN 3 THEN '3-Quarta-feira'
            WHEN 4 THEN '4-Quinta-feira'
            WHEN 5 THEN '5-Sexta-feira'
            WHEN 6 THEN '6-Sábado'
        END AS Dia_da_Semana,
        
        COUNT(*) AS Total_Acidentes,
        SUM(CASE WHEN INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) AS Acidentes_Fatais,
        ROUND(
            (SUM(CASE WHEN INDICADOR_FATALIDADE = 'SIM' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 
        2) AS Taxa_Letalidade
        
    FROM tabela_acidentes
    GROUP BY Dia_da_Semana
    ORDER BY Dia_da_Semana ASC;
"""

print("--- DISTRIBUIÇÃO DE ACIDENTES POR DIA DA SEMANA ---")
resultado_dias = pd.read_sql_query(query_dias_semana, conexao)

# Um pequeno truque no Pandas para limpar o número que usamos para ordenar no SQL
resultado_dias['Dia_da_Semana'] = resultado_dias['Dia_da_Semana'].str.split('-').str[1]

print(resultado_dias.to_string(index=False))

--- DISTRIBUIÇÃO DE ACIDENTES POR DIA DA SEMANA ---
Dia_da_Semana  Total_Acidentes  Acidentes_Fatais  Taxa_Letalidade
      Segunda             1604                10             0.62
        Terça             1624                11             0.68
       Quarta             1660                13             0.78
       Quinta             1588                 6             0.38
        Sexta             1934                21             1.09
       Sábado             1542                22             1.43
      Domingo             1170                24             2.05


Conclusão Analítica e Plano de Ação
A análise cruzada entre os dados de ocorrências viárias, demografia e mortalidade geral de Belo Horizonte (2021) nos permitiu ir além dos números absolutos e extrair insights direcionáveis.

📌 Principais Descobertas (Insights)
O Risco Real: A taxa de letalidade e a proporção de acidentes por 100 mil habitantes demonstram que, embora o trânsito não seja a principal causa de mortalidade geral (especialmente em um ano atípico impactado pela pandemia de COVID-19), ele representa um risco diário e constante para a população.

Sazonalidade e Comportamento: A análise temporal revelou que dias específicos da semana, como Sábado / Domingo, concentram os maiores índices de severidade, indicando uma forte correlação entre os dias de descanso/lazer e a imprudência nas vias.

Tipologia Crítica: Tipos específicos de acidentes, como CHOQUE MECANICO e COLISAO DE VEICULOS, destacaram-se com as maiores taxas de letalidade, exigindo atenção prioritária.

🎯 Recomendações e Próximos Passos
Com base nos dados levantados, sugerem-se as seguintes frentes de atuação para o poder público e iniciativas privadas:

Alocação Inteligente de Recursos: Direcionar blitzes, viaturas de fiscalização e equipes de resgate de forma estratégica nos dias da semana e meses que apresentaram os maiores picos de ocorrências.

Intervenção na Infraestrutura: Realizar auditorias de segurança viária focadas em prevenir os tipos de acidentes identificados como os mais letais nas consultas SQL.

Educação e Impacto Social: Estes insights não servem apenas para painéis gerenciais. Eles podem fundamentar iniciativas lúdicas e de conscientização de alto impacto — como o desenvolvimento de jogos educativos e simulações que ensinem regras de circulação para crianças e adolescentes. Transformar essas estatísticas brutas em uma verdadeira missão por um trânsito seguro engaja a nova geração e ataca o problema na base estrutural da educação.

Ferramentas Utilizadas neste Projeto: Python (Pandas, Matplotlib), Great Expectations (Qualidade de Dados) e SQLite (Modelagem Relacional e Consultas).